In [1]:
import os
import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patheffects as pe
import seaborn as sns
from rapidfuzz import fuzz, process
from sklearn.model_selection import cross_validate
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed

from scholarlm.utils import (
    load_and_process_results,
    match_datasets,
    matching_precision_recall,
    get_filenames_in_directory,
    fit_temperature,
    apply_temperature,
    fit_temperature_from_probs,
    apply_temperature_from_probs,
)
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

/projectnb/mcnet/kevin/coastal/scholarlm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Ground Truth Dataset

In [2]:
# ---------------------------------
# Load from ground truth dataset
# ---------------------------------

# Directory
with open(os.path.join("../data/nfix/directory.json"), "r") as f:
    paper_info = json.load(f)

def text_or_table_extraction(location):
    if 'figure' in location:
        return False
    if 'supplement' in location:
        return False
    if 'archive' in location:
        return False
    if 'author' in location:
        return False
    else:
        return True

registered_paper_info = {
    R: Rinfo for R,Rinfo in paper_info.items() if text_or_table_extraction(Rinfo['extraction_location']) 
}
registered_ids = list(registered_paper_info.keys())

#ground_truth_df = pd.read_csv("../data/nfix/nfix_data_corrected.csv", encoding_errors='ignore', index_col = 0)
#ground_truth_df = ground_truth_df.loc[ground_truth_df.title.isin(registered_titles)]
#ground_truth_df = ground_truth_df.reset_index(drop=True)

ground_truth_df = pd.read_csv("../data/nfix/meta/aquatic_N2fix_rates.csv")
ground_truth_df = ground_truth_df.loc[ground_truth_df.reference_id.isin(registered_ids)]

id_cols = [
        'nfix_rate_id', 'reference_id', 'site_name', 'latitude', 'longitude',
        'habitat', 'year', 'month', 'day', 'hour_minute', 'season',
        'substrate', 'substrate_details'
]

# Build a list of records, one per attribute
records = []

# 1) nfix_rate_converted -> attribute="rate_converted"
#    error from nfix_error_converted, error_type from nfix_error_type,
#    units from nfix_unit_converted
'''
records.append(nfix_df[id_cols].assign(
    attribute='rate_converted',
    value=nfix_df['nfix_rate_converted'],
    error=nfix_df['nfix_error_converted'],
    error_type=nfix_df['nfix_error_type'],
    units=nfix_df['nfix_unit_converted'],
))
'''

# 2) nfix_rate_original -> attribute="rate_original"
#    error from nfix_error_original, error_type from nfix_error_type,
#    units from nfix_unit_original
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_rate',
    value=ground_truth_df['nfix_rate_original'],
    error=ground_truth_df['nfix_error_original'],
    error_type=ground_truth_df['nfix_error_type'],
    units=ground_truth_df['nfix_unit_original'],
))

# 3) sample_depth -> attribute="sample_depth"
records.append(ground_truth_df[id_cols].assign(
    attribute='sample_depth',
    value=ground_truth_df['sample_depth'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

# 4) nfix_method -> attribute="method"
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_method',
    value=ground_truth_df['nfix_method'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

'''
# 5) nfix_incubation_time -> attribute="incubation_time"
records.append(nfix_df[id_cols].assign(
    attribute='incubation_time',
    value=nfix_df['nfix_incubation_time'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))
'''

# 6) nfix_incubation_temp -> attribute="incubation_temp"
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_incubation_temp_temperature',
    value=ground_truth_df['nfix_incubation_temp'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

ground_truth_df = pd.concat(records, ignore_index=True)

# Reorder columns
ground_truth_df = ground_truth_df[id_cols + ['attribute', 'value', 'error', 'error_type', 'units']]
ground_truth_df = ground_truth_df.dropna(subset=['value'])

# Optionally sort so each original row's attributes are grouped together
ground_truth_df = ground_truth_df.sort_values(id_cols, kind='mergesort').reset_index(drop=True)

ground_truth_df = ground_truth_df.loc[ground_truth_df.attribute == 'nfix_rate']
ground_truth_df.reset_index(drop=True, inplace=True)

In [3]:
ground_truth_df

,nfix_rate_id,reference_id,site_name,latitude,longitude,habitat,year,month,day,hour_minute,season,substrate,substrate_details,attribute,value,error,error_type,units
0,N1,R231,south west coast of Australia,-28.000000,114.000000,continental shelves,2003.0,10.0,NaN,NaN,NaN,wc,water,nfix_rate,0.017,0.006,stdev.,nmol-n l-1 h-1
1,N1027,R25,"East Weeks Bay, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,1.6,0.200,std. err,umol-n m-2 h-1
2,N1028,R25,"Weeks Bay Mouth, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,2.4,0.700,std. err,umol-n m-2 h-1
3,N1029,R25,"West Weeks Bay, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,2.6,0.400,std. err,umol-n m-2 h-1
4,N1030,R109,"Pearl River Estuary, South China Sea",22.747341,113.652219,estuaries,2018.0,7.0,31.0,NaN,NaN,benthos,sediment - inner estuary,nfix_rate,0.19,0.110,stdev.,ug-N g-1 d-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,N983,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1990.0,7.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,22.7,NaN,NaN,nmol-c2h4 cm-2 h-1
987,N984,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1990.0,11.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,7.3,NaN,NaN,nmol-c2h4 cm-2 h-1
988,N985,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1991.0,3.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,20.3,NaN,NaN,nmol-c2h4 cm-2 h-1
989,N986,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1991.0,7.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,5.2,NaN,NaN,nmol-c2h4 cm-2 h-1


### Full, Extracted Dataset

In [11]:
# ---------------------------------
# Load experiment results
# ---------------------------------

#experiment_data_path = "../data/experiments/2026_04_01/nfix_final.json"
experiment_data_path = "../data/experiments/nfix/extraction/gemma-3-27b/2026_04_08/final.json"

unit_conversion_table = {
    'nfix_rate': {},
    'nfix_incubation_time': {},
    'nfix_incubation_temperature': {"°C": 1,},
}

attribute_types = {
    'nfix_rate_mass': float,
    'nfix_rate_areal': float,
    'nfix_rate_volumetric': float,
    'nfix_incubation_time': float,
    'nfix_incubation_temperature': float,
}

# NOTE: some of these things you should get rid of in your extraction process!
drop_keys = ["feature_terms", "attribute_terms", "abbreviations", "table_logprob", "page_logprob", "judgement_raw_text"]
drop_attrs = ['nfix_incubation_time', 'nfix_incubation_temperature', 'sample_depth']

extracted_df = load_and_process_results(
    json_path=experiment_data_path,
    unit_conversion_table=unit_conversion_table,
    attribute_types=attribute_types,
    drop_keys=drop_keys,
    drop_attrs=drop_attrs,
    attribute_col="attribute",
    value_col="value",
    unit_col="units",
    out_col="processed_value"
)



# NOTE you need to change this to 'attribute'
#extracted_df.rename(columns={"feature": "attribute"}, inplace=True)
extracted_df.sort_values(by=["title", "attribute"], inplace=True)
extracted_df.reset_index(drop=True, inplace=True)
extracted_df.attribute = "nfix_rate"

#xtracted_df = extracted_df.loc[extracted_df.attribute == 'nfix_rate']
#extracted_df = extracted_df.reset_index(drop=True)

In [12]:
extracted_df

,reference,doi,doi_data,url,publication_year,authors,title,publication,volume,pages,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,processed_value
0,Montoya et al 1996 AEM 62,10.1128/aem.62.3.986-993.1996,NaN,https://journals.asm.org/doi/10.1128/aem.62.3....,1996,"Montoya, J P; Voss, M; Kahler, P; Capone, D G","A Simple, High-Precision, High-Sensitivity Tra...",Applied and Environmental Microbiology,62,986-993,...,3.440,nmol/ml/h,[2],[1],"[('Concentrated', '18 July 1994', 'On deck, 10...",[c2h2_reduction_mean_nmol_ml_h],[table],"[<table number=""1"">\n<caption>Effects of bottl...",2716,3.440
1,Hicks and Silvester 1990 N Zealand J Mar Fresh...,10.1080/00288330.1990.9516439,NaN,http://www.tandfonline.com/doi/abs/10.1080/002...,1990,"Hicks, Brendan J.; Silvester, Warwick B.",Acetylene reduction associated with Zostera no...,New Zealand Journal of Marine and Freshwater R...,24,481-486,...,15.200,μmol C₂H₄ m⁻² h⁻¹,[1],[None],[None],[None],[text],[Acetylene reduction associated with Zostera n...,2,15.200
2,Hicks and Silvester 1990 N Zealand J Mar Fresh...,10.1080/00288330.1990.9516439,NaN,http://www.tandfonline.com/doi/abs/10.1080/002...,1990,"Hicks, Brendan J.; Silvester, Warwick B.",Acetylene reduction associated with Zostera no...,New Zealand Journal of Marine and Freshwater R...,24,481-486,...,148.000,kg N ha⁻¹ y⁻¹,[5],[None],[None],[None],[text],[predominantly with leaves of Thalassia testud...,3,148.000
3,Hicks and Silvester 1990 N Zealand J Mar Fresh...,10.1080/00288330.1990.9516439,NaN,http://www.tandfonline.com/doi/abs/10.1080/002...,1990,"Hicks, Brendan J.; Silvester, Warwick B.",Acetylene reduction associated with Zostera no...,New Zealand Journal of Marine and Freshwater R...,24,481-486,...,15.200,umol C₂H₄ m⁻² h⁻¹,[3],[1],"[('Zostera', 'With plants')]",[acetylene_reduction_mean_umol_c2h4_m2_h1],[table],"[<table number=""1"">\n<caption>Table 1 Acetylen...",4,15.200
4,Hicks and Silvester 1990 N Zealand J Mar Fresh...,10.1080/00288330.1990.9516439,NaN,http://www.tandfonline.com/doi/abs/10.1080/002...,1990,"Hicks, Brendan J.; Silvester, Warwick B.",Acetylene reduction associated with Zostera no...,New Zealand Journal of Marine and Freshwater R...,24,481-486,...,15.200,μmol C₂H₄ m⁻² h⁻¹,[1],[None],[None],[None],[text],[Acetylene reduction associated with Zostera n...,5,15.200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2336,Twomey et al 2007 Deep Sea Res 54,10.1016/j.dsr2.2006.10.001,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2007,"Twomey, L.J.; Waite, A.M.; Pez, V.; Pattiaratc...",Variability in nitrogen uptake and fixation in...,Deep Sea Research Part II: Topical Studies in ...,54,925-942,...,0.000,μM N h-1,[2],[None],[None],[None],[text],[depending on the continental shelf width (tot...,1242,0.000
2337,Twomey et al 2007 Deep Sea Res 54,10.1016/j.dsr2.2006.10.001,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2007,"Twomey, L.J.; Waite, A.M.; Pez, V.; Pattiaratc...",Variability in nitrogen uptake and fixation in...,Deep Sea Research Part II: Topical Studies in ...,54,925-942,...,1.900,nmol L⁻¹ h⁻¹,[5],[None],[None],[None],[text],"[(P = 0.087, n = 42), confirmed by poor linear...",1243,1.900
2338,Twomey et al 2007 Deep Sea Res 54,10.1016/j.dsr2.2006.10.001,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2007,"Twomey, L.J.; Waite, A.M.; Pez, V.; Pattiaratc...",Variability in nitrogen uptake and fixation in...,Deep Sea Research Part II: Topical Studies in ...,54,925-942,...,0.030,nmol L⁻¹ h⁻¹,[9],[None],[None],[None],[text],[Fig. 7. Nitrogen fixation measured at N uptak...,1244,0.030
2339,Twomey et al 2007 Deep Sea Res 54,10.1016/j.dsr2.2006.10.001,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2007,"Twomey, L.J.; Waite, A.M.; Pez, V.; Pattiaratc...",Variability in nitrogen uptake and fixation in...,Deep Sea Research Part II: Topical Studies in ...,54,925-942,...,15.000,nmol L⁻¹ h⁻¹,[11],[None],[None],[None

### Match Extractions to Ground Truth

In [29]:
# Set of attributes which must be strictly equivalent to create a match
strict_matching = {
    "reference_id": "paper_code", # name in the ground truth dataset : name in the extracted dataset
    "attribute": "attribute",
    "value": "value"
}

# Set of attributes which should be 
# compared by a fuzzy matching (roughly similar) to create a match.
fuzzy_matching = {
    "site_name": "name",
    #"substrate": "substrate_type",
    #"habitat": "habitat_type",
}

# This can take a while to run if you have a lot of data, 
# since it compares every extracted row to every ground truth row.
matching, matching_recall, matching_precision = matching_precision_recall(
    ground_truth_df,
    extracted_df,
    strict_matching=strict_matching,
    fuzzy_matching=fuzzy_matching,
    fuzzy_threshold = 0.0
)

print(f"Recall: {matching_recall:.4f}")
print(f"Precision: {matching_precision:.4f}")

Recall: 0.2775
Precision: 0.1175


### Debugging

In [30]:
gt_matched = np.array([False] * ground_truth_df.shape[0])
ex_matched = np.array([False] * extracted_df.shape[0])
for gt_idx, ex_idx in matching:
    gt_matched[gt_idx] = True
    ex_matched[ex_idx] = True

unmatched_gt = np.where(~gt_matched)[0]
unmatched_ex = np.where(~ex_matched)[0]

matched_gt_df = ground_truth_df[gt_matched == True]
unmatched_gt_df = ground_truth_df[gt_matched == False]
unmatched_gt_refs = unmatched_gt_df.reference_id.value_counts().index

matched_ex_df = extracted_df[ex_matched == True]
unmatched_ex_df = extracted_df[ex_matched == False]
unmatched_ex_refs = unmatched_ex_df.paper_code.value_counts().index

In [31]:
unmatched_gt_df.reference_id.value_counts()

reference_id
R163    103
R164     82
R172     37
R51      29
R248     28
       ... 
R76       1
R96       1
R116      1
R82       1
R99       1
Name: count, Length: 88, dtype: int64

In [32]:
ref = "R51"
gt_ref_df = ground_truth_df.loc[ground_truth_df.reference_id == ref]
unmatched_gt_ref_df = unmatched_gt_df.loc[unmatched_gt_df.reference_id == ref]
ex_ref_df = extracted_df.loc[extracted_df.paper_code == ref]
unmatched_ex_ref_df = unmatched_ex_df.loc[unmatched_ex_df.paper_code == ref]

In [36]:
unmatched_gt_ref_df

,nfix_rate_id,reference_id,site_name,latitude,longitude,habitat,year,month,day,hour_minute,season,substrate,substrate_details,attribute,value,error,error_type,units
790,N4045,R51,Castle Forbes - Upper,-43.1402,146.976700,tidal flats,2002.0,2.0,3.0,NaN,summer,benthos,sediment,nfix_rate,78.49,NaN,NaN,umol-n m-2 h-1
791,N4046,R51,Castle Forbes - Upper,-43.1402,146.976700,tidal flats,2002.0,2.0,3.0,NaN,summer,benthos,sediment,nfix_rate,41.37,NaN,NaN,umol-n m-2 h-1
792,N4047,R51,Castle Forbes - Upper,-43.1402,146.976700,tidal flats,2002.0,2.0,13.0,NaN,summer,benthos,sediment,nfix_rate,253.14,NaN,NaN,umol-n m-2 h-1
793,N4048,R51,Castle Forbes - Upper,-43.1402,146.976700,tidal flats,2002.0,4.0,NaN,NaN,autumn,benthos,sediment,nfix_rate,1.217,NaN,NaN,umol-n m-2 h-1
794,N4049,R51,Castle Forbes - Upper,-43.1402,146.976700,tidal flats,2002.0,4.0,NaN,NaN,autumn,benthos,sediment,nfix_rate,1.825,NaN,NaN,umol-n m-2 h-1
798,N4052,R51,Castle Forbes - Upper,-43.1402,146.976700,tidal flats,2002.0,10.0,NaN,NaN,spring,benthos,sediment,nfix_rate,1.83,NaN,NaN,umol-n m-2 h-1
799,N4053,R51,Castle Forbes - Upper,-43.1402,146.976700,tidal flats,2002.0,10.0,NaN,NaN,spring,benthos,sediment,nfix_rate,2.43,NaN,NaN,umol-n m-2 h-1
800,N4054,R51,Castle Forbes - Lower,-43.1402,146.976700,tidal flats,2002.0,2.0,NaN,NaN,summer,benthos,sediment,nfix_rate,52.65,NaN,NaN,umol-n m-2 h-1
801,N4055,R51,Castle Forbes - Lower,-43.1402,146.976700,tidal flats,2002.0,2.0,NaN,NaN,summer,benthos,sediment,nfix_rate,12.24,NaN,NaN,umol-n m-2 h-1
802,N4056,R51,Castle Forbes - Lower,-43.1402,146.976700,tidal flats,2002.0,4.0,NaN,NaN,autumn,benthos,sediment,nfix_rate,46.53,NaN,NaN,umol-n m-2 h-1


In [42]:
unmatched_ex_ref_df.iloc[:,-11:]

,attribute,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,processed_value
111,nfix_rate,250.0,μmol m⁻² h⁻¹,[6],[None],[None],[None],[text],[Fig. 2. Rates of nitrogen fixation measured b...,1976,250.0
112,nfix_rate,74.0,µmol m⁻² h⁻¹,[9],[None],[None],[None],[text],[ing activity of fauna (bio-irrigation) or the...,1977,74.0
113,nfix_rate,5.0,μmol N m⁻² h⁻¹,[12],[None],[None],[None],[text],"[sediment, suggesting that assimilation of NO\...",1978,5.0
114,nfix_rate,25.0,mmol N m⁻² d⁻¹,[7],[1],"[('CF', 'upper', 'Autumn')]",[n2_fixation_rate_2002_mmol_N_m2_d1],[table],"[<table number=""1"">\n<caption>Estimated rates ...",1979,25.0
115,nfix_rate,250.0,µmol m⁻² h⁻¹,[6],[None],[None],[None],[text],[Fig. 2. Rates of nitrogen fixation measured b...,1982,250.0
116,nfix_rate,74.0,μmol m⁻² h⁻¹,[9],[None],[None],[None],[text],[ing activity of fauna (bio-irrigation) or the...,1983,74.0
117,nfix_rate,5.0,µmol N m⁻² h⁻¹,[12],[None],[None],[None],[text],"[sediment, suggesting that assimilation of NO\...",1984,5.0
118,nfix_rate,25.0,mmol N m⁻² d⁻¹,[7],[1],"[('CF', 'upper', 'Autumn')]",[n2_fixation_rate_2002_mmol_N_m2_d1],[table],"[<table number=""1"">\n<caption>Estimated rates ...",1985,25.0
119,nfix_rate,250.0,μmol m⁻² h⁻¹,[6],[None],[None],[None],[text],[Fig. 2. Rates of nitrogen fixation measured b...,1988,250.0
120,nfix_rate,74.0,µmol m⁻² h⁻¹,[9],[None],[None],[None],[text],[ing activity of fauna (bio-irrigation) or the...,1989,74.0


In [35]:
len(unmatched_ex_ref_df)

30

In [43]:
table1_unmatched_df = unmatched_ex_ref_df[unmatched_ex_ref_df['table_number'].apply(lambda x: 1 in x)]

In [50]:
table1_unmatched_df.loc[table1_unmatched_df.value == 3.7].iloc[:, 14:]

,name,location,latitude,longitude,habitat_type,substrate_type,sample_depth,year,month,day,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,processed_value
1098,Mid-Atlantic Shelf,"37.519°N, −75.050°W",37.519,-75.050,shelf,water,29,2006.0,7.0,NaN,...,3.7,unitless,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",704,3.7
1105,Mid-Atlantic Shelf,"38.042°N, −74.299°W",38.042,-74.299,shelf,water,None,2006.0,7.0,NaN,...,3.7,umol N m−2 d−1,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",727,3.7
1135,Mid-Atlantic Shelf,"38.042°N, −74.299°W",38.042,-74.299,shelf,water,None,2006.0,7.0,NaN,...,3.7,3.7,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",886,3.7
1149,Mid-Atlantic Shelf,"37.695°N, −75.298°W",37.695,-75.298,shelf,water,None,2006.0,10.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",935,3.7
1156,Mid-Atlantic Shelf,"37.631°N, −75.152°W",37.631,-75.152,shelf,water,None,2006.0,10.0,NaN,...,3.7,3.7,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",957,3.7
1163,Mid-Atlantic Shelf,"37.519°N, −75.050°W",37.519,-75.050,shelf,water,None,2006.0,10.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",981,3.7
1199,Mid-Atlantic Shelf,"36.554°N, −75.706°W",36.554,-75.706,shelf,water,None,2009.0,8.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1117,3.7
1206,Mid-Atlantic Shelf,"37.428°N, −75.476°W",37.428,-75.476,shelf,water,None,2009.0,8.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1142,3.7
1230,Mid-Atlantic Shelf,"36.417°N, −75.700°W",36.417,-75.700,shelf,water,None,2009.0,8.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1261,3.7
1237,Mid-Atlantic Shelf,"36.417°N, −75.700°W",36.417,-75.700,shelf,water,None,2009.0,8.0,NaN,...,3.7,umol N m−2 d−1,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1285,3.7


In [46]:
table1_unmatched_df.units.value_counts()

units
nmol N L−1 d−1    28
3.7               17
umol N m−2 d−1    13
nmol N            10
umol-N m-2 d-1     5
n2_fixation        1
unitless           1
3.8                1
nmol/L/d           1
nmol L-1 d-1       1
nmol L−1 d−1       1
Name: count, dtype: int64